# Imports

In [1]:
import os
import json
from dotenv import load_dotenv
from langchain_core.tools import tool
from langchain_core.documents import Document
from langchain_pinecone import PineconeVectorStore
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# Pinecone Upload

In [2]:
# Load environment variables
load_dotenv()

# --- 1. CONFIGURATION ---
# Using gemini-embedding-001 (768 dimensions) as per official stable docs
INDEX_NAME = "youtube-agent-musical"
JSON_PATH = "../data/agent_db/godzilla_master_context.json"

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001", 
    google_api_key=os.getenv("GEMINI_API_KEY"),
    task_type="retrieval_document" # Optimized for storage/indexing
)

print(f"🚀 Initializing ingestion for index: {INDEX_NAME}")

try:
    # --- 2. DATA PREPARATION ---
    # Loading the refined master context (Demucs + Whisper logic)
    with open(JSON_PATH, "r", encoding="utf-8") as f:
        agent_data = json.load(f)
    
    documents = []
    for segment in agent_data:
        text = segment.get("text", "").strip()
        
        # Only process segments with actual content
        if text:
            doc = Document(
                page_content=text,
                metadata={
                    "start": segment["start"],
                    "end": segment["end"],
                    "is_music_piece": segment["is_music_piece"]
                }
            )
            documents.append(doc)
            
    print(f"📄 Prepared {len(documents)} documents for vectorization.")

    # --- 3. PINECONE UPLOAD ---
    # Syncing vectors to the 768-dimension Gemini-compatible index
    vectorstore = PineconeVectorStore.from_documents(
        documents, 
        embedding=embeddings,
        index_name=INDEX_NAME,
        pinecone_api_key=os.getenv("PINECONE_API_KEY")
    )
    
    print("-" * 50)
    print("✅ SUCCESS: Ingestion complete with stable gemini-embedding-001!")
    print(f"Knowledge base is live in index: {INDEX_NAME}")
    print("-" * 50)

except Exception as e:
    print(f"❌ Critical Error during ingestion: {e}")

🚀 Initializing ingestion for index: youtube-agent-musical
📄 Prepared 56 documents for vectorization.
--------------------------------------------------
✅ SUCCESS: Ingestion complete with stable gemini-embedding-001!
Knowledge base is live in index: youtube-agent-musical
--------------------------------------------------
